# 01 · Data Preparation & EDA
NYC Yellow and Green taxi trips, January–June 2026.
Goal: audit the raw files, clean them, build the trip-duration target, and save processed data.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
SEED = 33
np.random.seed(SEED)

In [3]:
# DISPLAY
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [6]:
con = duckdb.connect()

In [4]:
# Paths (relative to the notebooks/ folder)
ROOT = Path("..")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

YELLOW_FILES = str(RAW / "yellow" / "yellow_tripdata_2026-*.parquet")
GREEN_FILES = str(RAW / "green" / "green_tripdata_2026-*.parquet")
ZONES_FILE = RAW / "taxi_zone_lookup.csv"

In [12]:
#Row counts
con.sql(f"SELECT COUNT(*) AS yellow_rows FROM '{YELLOW_FILES}'").df()

,yellow_rows
0,22836530


In [13]:
con.sql(f"SELECT COUNT(*) AS green_rows FROM '{GREEN_FILES}'").df()

,green_rows
0,255175


In [10]:
# Columns and types
con.sql(f"DESCRIBE SELECT * FROM '{YELLOW_FILES}'").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [11]:
con.sql(f"DESCRIBE SELECT * FROM '{GREEN_FILES}'").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [14]:
# Summary of every column (counts, min, max, nulls)
con.sql(f"SUMMARIZE SELECT * FROM '{GREEN_FILES}'").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.3328813559322032,1.284346233493969,2,2,2,255175,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 23:05:50,2026-07-01 03:21:24,242176,2026-04-03 14:16:18.833193,NaN,2026-02-17 23:28:42.176804,2026-04-05 07:19:08.814902,2026-05-18 01:11:17.924978,255175,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:31:29,2026-07-01 18:14:34,251143,2026-04-03 14:37:31.084687,NaN,2026-02-17 22:14:20.104794,2026-04-05 07:04:03.831397,2026-05-18 05:52:16.653594,255175,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,255175,14.12
4,RatecodeID,BIGINT,1,99,7,1.239194516947451,0.9841999999050745,1,1,1,255175,14.12
5,PULocationID,INTEGER,1,265,247,97.35740178309004,56.517009792835886,74,75,106,255175,0.00
6,DOLocationID,INTEGER,1,265,255,142.6491740962085,77.20024267674381,74,140,229,255175,0.00
7,passenger_count,BIGINT,0,9,10,1.3084719002683118,0.9710451474210126,1,1,1,255175,14.12
8,trip_distance,DOUBLE,0.0,179830.92,2840,14.833881963358493,937.0473517026429,1.2471727382970128,2.0403975699191577,3.6411224087822047,255175,0.00
9,fare_amount,DOUBLE,-250.08,1676.7,4320,16.717533379053485,17.39538835945694,8.602843933224587,13.076115384682664,19.511045400808325,255175,0.00


In [15]:
con.sql(f"SUMMARIZE SELECT * FROM '{YELLOW_FILES}'").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.882208023723394,0.719029520062424,2,2,2,22836530,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-06-30 23:59:59,10176022,2026-04-02 22:36:20.712294,NaN,2026-02-16 15:45:46.250192,2026-04-04 06:14:59.410015,2026-05-17 06:11:25.570378,22836530,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-07-01 22:33:25,10556917,2026-04-02 22:54:08.074988,NaN,2026-02-16 05:09:32.22114,2026-04-03 14:09:08.411239,2026-05-17 12:06:46.591586,22836530,0.00
3,passenger_count,BIGINT,0,9,10,1.2439878312243728,0.6467090181088792,1,1,1,22836530,25.51
4,trip_distance,DOUBLE,0.0,328522.2,7648,5.648561099693675,563.2281204172576,1.0172746778166708,1.8333886844184237,3.734254456705423,22836530,0.00
5,RatecodeID,BIGINT,1,99,7,4.774999162294431,18.62159983861536,1,1,1,22836530,25.51
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,22836530,25.51
7,PULocationID,INTEGER,1,265,262,161.84268765876428,66.8258491188188,117,161,233,22836530,0.00
8,DOLocationID,INTEGER,1,265,262,161.4179360874879,70.74686235473871,109,162,234,22836530,0.00
9,payment_type,BIGINT,0,5,6,0.8662002940026352,0.645259908239611,0,1,1,22836530,0.00
